# RE_303 · Es cancel·larà aquesta reserva? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Ets la persona de dades d'una petita cadena amb **dos hotels a Portugal**, un de ciutat i un de
platja. Una de cada tres reserves acaba cancel·lada, i la direcció vol fer *overbooking* amb cap:
quan entra una reserva nova, saber si és probable que es cancel·li.

**La pregunta:** quin encert (*accuracy*) es pot prometre a la direcció, amb un model que funcioni **el dia que
entra la reserva**? Han de poder-se'l creure, perquè hi posaran clients al carrer si s'equivoca.

L'enunciat complet, amb el diccionari de columnes, és a [RE_303](../RE_303_hotels_cancellacions.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

num = make_column_selector(dtype_include="number")
txt = make_column_selector(dtype_exclude="number")
def prepara():
    return make_column_transformer(
        (SimpleImputer(strategy="median"), num),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore", min_frequency=20)), txt))

def parteix(d):
    X, y = d.drop(columns="is_canceled"), d["is_canceled"]
    return train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

## 1.1 · La foto

Forma, tipus i nuls. Quina proporció de reserves es cancel·len? Quina *accuracy* tindria un
model que digués sempre la classe més freqüent?

In [ ]:
print(df.dtypes.value_counts().to_dict())
print(df.isna().sum()[lambda s: s > 0].to_dict())
prop = df["is_canceled"].value_counts(normalize=True)
assert np.isclose(prop.sum(), 1)
print(prop.round(3).to_dict(), "-> referència:", round(prop.max(), 3))

Es cancel·la el **37 %** de les reserves: la referència (dir sempre «no») encerta un 63 %.

## 1.2 · Files repetides

Quantes files són idèntiques a una altra? Mira-ne unes quantes. Poden ser reserves diferents que
coincideixen en tot, o són un error? Tingues en compte quines columnes **no** hi ha a la taula.

In [ ]:
n_dup = df.duplicated().sum()
assert n_dup + len(df.drop_duplicates()) == len(df) and n_dup == 31994
print("duplicats:", n_dup, f"({n_dup / len(df):.0%})")
print(df.groupby(df.duplicated(keep=False))["is_canceled"].mean().round(3))

Hi ha **31.994 files idèntiques** a una altra (27 %). No hi ha cap identificador de reserva ni
de client, així que poden ser reserves de grup reals (mateix dia, mateix preu, mateix agent...)
o registres duplicats; **no es pot saber**. Les repetides es cancel·len molt més (58 % contra
26 %): pesen en el model.

## 2.1 · El primer model, sense pensar

Fes servir **totes** les columnes (menys l'objectiu) en un `Pipeline` amb imputació i one-hot, i
entrena un arbre de decisió de profunditat 3. Quina *accuracy* treu al test? Compara-la amb la
referència de la 1.1. És creïble?

In [ ]:
X_tr, X_te, y_tr, y_te = parteix(df)
arbre = make_pipeline(prepara(), DecisionTreeClassifier(max_depth=3, random_state=0)).fit(X_tr, y_tr)
acc_ingenu = arbre.score(X_te, y_te)
print("accuracy:", round(acc_ingenu, 4))
print(export_text(arbre[-1], feature_names=list(arbre[:-1].get_feature_names_out())))
assert acc_ingenu > 0.99

Un arbre de **profunditat 3** encerta pràcticament el 100 % de les reserves. Un problema de
negoci real no es resol amb tres preguntes: és massa bo per ser cert. L'arbre només fa servir
`reservation_status`: amb **una sola pregunta** n'hi ha prou.

## 2.2 · Què sap el model que no hauria de saber

Fes la pregunta columna per columna: **aquest valor, el sé el dia que entra la reserva?** Llegeix
el diccionari de dades. Fes una llista de les columnes que no es poden fer servir i, per a cada
una, escriu en una línia per què. Comprova la teva sospita amb una taula creuada contra
l'objectiu.

In [ ]:
print(pd.crosstab(df["reservation_status"], df["is_canceled"]))
print(pd.crosstab(df["reserved_room_type"] != df["assigned_room_type"], df["is_canceled"], normalize="columns").round(3))

In [ ]:
FUITA = {
    "reservation_status": "estat final (Check-Out / Canceled / No-Show): ÉS l'objectiu escrit d'una altra manera",
    "reservation_status_date": "data de l'últim estat: es coneix quan la reserva ja ha acabat o s'ha cancel·lat",
    "assigned_room_type": "l'habitació que et donen en arribar; a les cancel·lades gairebé sempre és la reservada",
    "booking_changes": "canvis fets fins al check-in o la cancel·lació: s'acumulen després (discutible)",
}
neta = df.drop(columns=list(FUITA))
X_tr, X_te, y_tr, y_te = parteix(neta)
assert set(FUITA) & set(X_tr.columns) == set()
arbre2 = make_pipeline(prepara(), DecisionTreeClassifier(max_depth=3, random_state=0)).fit(X_tr, y_tr)
print("arbre de profunditat 3 sense fuites:", round(arbre2.score(X_te, y_te), 4))
assert arbre2.score(X_te, y_te) < acc_ingenu

`reservation_status` és la resposta (Canceled + No-Show = cancel·lada, Check-Out = no) i la seva
data només existeix quan la història ja ha acabat. `assigned_room_type` es decideix en arribar
(a les cancel·lades coincideix amb la reservada en el 98 % dels casos, perquè ningú no l'ha
canviat). `booking_changes` és discutible: es poden acceptar com a «canvis fins avui», però al
dataset són els finals. Sense elles, el mateix arbre baixa a un 76 %.

## 2.3 · Els duplicats i el test

Parteix en entrenament i test **sense** treure els duplicats. Quina proporció de files del test
té una còpia exacta a l'entrenament? Després entrena un bosc aleatori dues vegades —amb duplicats
i sense— i compara l'*accuracy* de test. Quina diferència hi ha, i per què?

In [ ]:
fusio = X_te.merge(X_tr.drop_duplicates(), how="left", indicator=True)
prop_copia = (fusio["_merge"] == "both").mean()
assert 0 < prop_copia < 1
print(f"files del test amb una còpia exacta a l'entrenament: {prop_copia:.1%}")

In [ ]:
def bosc():
    return make_pipeline(prepara(), RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1))

amb = bosc().fit(X_tr, y_tr)
acc_amb = amb.score(X_te, y_te)

unica = neta.drop_duplicates()
assert unica.duplicated().sum() == 0
Xu_tr, Xu_te, yu_tr, yu_te = parteix(unica)
assert list(Xu_tr.columns) == list(X_tr.columns)
sense = bosc().fit(Xu_tr, yu_tr)
acc_sense = sense.score(Xu_te, yu_te)
ref_sense = DummyClassifier().fit(Xu_tr, yu_tr).score(Xu_te, yu_te)
print(f"bosc amb duplicats: {acc_amb:.3f}   sense duplicats: {acc_sense:.3f}")
assert acc_amb > acc_sense

Un **33 %** de les files del test tenen una còpia exacta a l'entrenament: per a aquestes el bosc
no prediu, **recorda**. Amb duplicats l'*accuracy* és d'un 89 %; sense, d'un 85 %. Els 4 punts
de diferència són memòria, no aprenentatge. Si són reserves de grup reals, la solució correcta
seria partir per grups; com que no hi ha identificador, el més honest és treure'ls.

## 2.4 · El número honest

Amb les decisions de la 2.2 i la 2.3, entrena el model final i dona'n l'*accuracy* de test, al
costat de la referència (que també canvia si has tret duplicats: torna-la a calcular).

In [ ]:
print(f"referència (sense duplicats): {ref_sense:.3f}")
print(f"bosc honest:                  {acc_sense:.3f}   (+{acc_sense - ref_sense:.3f})")
assert 0.80 < acc_sense < 0.88 and ref_sense > 0.70

**Resposta:** sense les columnes que es coneixen després i sense duplicats, un bosc aleatori
encerta un **85 %** de les reserves noves (partició aleatòria), contra un 72 % de la referència
(sense duplicats, la proporció de cancel·lades baixa al 28 %). El 100 % del primer model era
una fuita; el 89 % amb duplicats, memòria.

## 3.1 · Partir pel temps

Un model es fa servir amb reserves **futures**. Entrena amb les arribades de 2015 i 2016 i
avalua amb les de 2017. Quina *accuracy* surt, comparada amb la partició aleatòria? Quina és més
honesta per a l'encàrrec?

In [ ]:
u = unica
mask = u["arrival_date_year"] < 2017
Xt_tr, yt_tr = u[mask].drop(columns="is_canceled"), u.loc[mask, "is_canceled"]
Xt_te, yt_te = u[~mask].drop(columns="is_canceled"), u.loc[~mask, "is_canceled"]
temps = bosc().fit(Xt_tr, yt_tr)
acc_t = temps.score(Xt_te, yt_te)
ref_t = DummyClassifier().fit(Xt_tr, yt_tr).score(Xt_te, yt_te)
print(f"entrenar 2015-16, avaluar 2017: {acc_t:.3f}  (referència {ref_t:.3f})")

Partint pel temps l'*accuracy* baixa al **76 %** (referència del 2017: 68 %): el 2017 no s'assembla
del tot al 2015-2016. És el número que s'acosta més a l'ús real (predir reserves futures). Si
s'ha de donar un sol número a la direcció, és aquest, amb la millora sobre la referència al
costat.